# Shor's algorithm on real hardware: what factoring 15 does and does not prove

| | |
|---|---|
| **Level** | Intermediate to advanced |
| **Time** | About 90 minutes |
| **Prerequisites** | Modular arithmetic; phase estimation and order finding |
| **Default devices** | Rigetti Cepheus and IQM Garnet |
| **Hardware jobs** | 2 per device |
| **Approximate cost** | about 350 credits on Garnet at 1000 shots; Rigetti is billed by execution time, about 10 credits per job in our tests |
| **Hardware notes** | In our tests the compiled circuit put 0.48 (Rigetti Cepheus) and 0.98 (IQM Garnet) of its shots on correct answers, against 0.25 for random guessing. The full circuit gave 0.22 and 0.24, no better than random. |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*QUEST intermediate and advanced series: Cryptography and Security*

Shor's algorithm factors an $n$-bit integer in time polynomial in $n$. That would break RSA, and the same method breaks Diffie-Hellman and elliptic-curve cryptography. Every few years a paper reports a number factored on a quantum device: 15, then 21, then larger ones. Most of them did not run the algorithm as written.

The usual shortcut is compilation. If you already know the period the algorithm is meant to find, you can reduce the circuit to something a small device can run, and it will report the right factors. Smolin, Smith and Vargo showed in 2013 that with this shortcut, two coherent qubits are enough to "factor" any product of two distinct odd primes. The size of the number does not matter. The length of the period does.

This notebook builds order finding for $N = 15$ in three ways. The first derives every gate from modular arithmetic and uses no knowledge of the period; we call it the **honest circuit**. The second tunes the same circuit to this instance. The third is a **compiled circuit** of the kind many demonstrations use, which we then apply to a 2048-bit RSA modulus to show what it proves. We run the honest and compiled circuits on real devices, then use published resource estimates to see what factoring RSA-2048 would take.

**Learning objectives**

1. Reduce factoring to order finding and implement the classical part of the reduction.
2. Build controlled modular multiplication from the arithmetic, and check it against the classical map.
3. Recover the period from a phase estimate by continued fractions, and explain why half the outcomes are useless.
4. Show that a compiled circuit that already knows the period "factors" a 2048-bit modulus with a few qubits.
5. Measure how far the honest and compiled circuits land from their ideal output on current hardware.
6. Apply the Gidney-Ekera resource estimates to RSA-2048 and state what they assume.

**Background needed:** quantum phase estimation (the QPE notebook in this series), greatest common divisors, multiplicative order, and the Chinese remainder theorem. No cryptography background is assumed.


## From factoring to order finding

Shor's algorithm does not factor directly. It solves a different problem, and a classical reduction turns the answer into factors.

Pick $a$ with no common factor with $N$. The **order** of $a$ modulo $N$ is the smallest $r > 0$ with

$$a^r \equiv 1 \pmod N.$$

If $r$ is even, then $a^{r/2}$ is a square root of 1 modulo $N$, so

$$\left(a^{r/2} - 1\right)\left(a^{r/2} + 1\right) = a^r - 1 \equiv 0 \pmod N,$$

and $N$ divides the product on the left. If also $a^{r/2} \not\equiv -1 \pmod N$, neither factor is divisible by $N$ alone, so each shares part of $N$, and $\gcd(a^{r/2} - 1, N)$ is a non-trivial factor. Two things can go wrong: $r$ can be odd, or $a^{r/2}$ can be $-1$. For $N$ with at least two distinct odd prime factors, a random $a$ fails with probability at most $1/2$, so a few attempts are enough.

All of this is classical. The one step a classical computer cannot do quickly is finding $r$.

The quantum part finds $r$ as a period. Define

$$U_a \lvert y \rangle = \lvert a y \bmod N \rangle.$$

Its eigenvalues are $e^{2\pi i s / r}$ for $s = 0, 1, \dots, r-1$. We cannot prepare an eigenvector without knowing $r$, but we do not need to: $\lvert 1 \rangle$ is an equal superposition of all $r$ of them. Phase estimation on $\lvert 1 \rangle$ returns an estimate of $s/r$ for a random $s$, and continued fractions recover $r$.

So the quantum circuit is phase estimation with $U_a$ as the controlled operation. Nearly all the cost is in building controlled $U_a^{2^j}$, which is modular multiplication. That is also where shortcuts get taken.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
from math import gcd, log2
from fractions import Fraction

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

sim = AerSimulator(seed_simulator=42)
BASIS = ['cx', 'rz', 'sx', 'x']       # a generic superconducting basis, for gate counting

N = 15                                 # the number to factor
A = 7                                  # the base whose order we will find
n = N.bit_length()                     # work register width, 4 qubits

print(f"Setup complete. Factoring N = {N} with base a = {A}, work register {n} qubits.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS = 1000
QUEST_JOB_TAGS = {"quest": "crypto-shor"}   # labels this notebook's hardware jobs for QUEST usage statistics


## The classical part

Before any quantum circuit runs, an implementation rules out the easy cases. $N$ must not be even, not a perfect power, and not divisible by a small prime. It then draws $a$ at random and checks $\gcd(a, N)$, because a lucky draw factors $N$ with no quantum computer needed.

We run those checks here, so that the base given to the circuit is one that genuinely needs its period found.

In [ ]:
def classical_precheck(N):
    """The tests a real implementation runs before touching a quantum device."""
    if N % 2 == 0:
        return f"even, factor 2"
    for b in range(2, int(log2(N)) + 1):
        root = round(N ** (1 / b))
        for cand in (root - 1, root, root + 1):
            if cand > 1 and cand ** b == N:
                return f"perfect power: {cand}^{b}"
    return None


print(f"N = {N}: precheck says {classical_precheck(N) or 'no easy factorisation, proceed'}")
print(f"gcd(a, N) = {gcd(A, N)}  (must be 1, else a already factors N)")

# The order, computed classically. We use this only to check the quantum answer,
# never to build the circuit.
r_true = next(r for r in range(1, N) if pow(A, r, N) == 1)
print(f"\nclassical order of {A} mod {N}: r = {r_true}")
print(f"a^(r/2) mod N = {pow(A, r_true // 2, N)}  (must not be N-1 = {N - 1})")
print(f"factors from the reduction: {gcd(pow(A, r_true // 2, N) - 1, N)} "
      f"and {gcd(pow(A, r_true // 2, N) + 1, N)}")

`r_true` is used in this notebook only to check the quantum result; the circuit construction below never reads it. Once $r$ is known the rest is easy arithmetic, which is why the security question comes down to whether a device can find $r$.

## Modular multiplication is a permutation

$U_a \lvert y \rangle = \lvert ay \bmod N \rangle$ maps basis states to basis states. Since $\gcd(a, N) = 1$, multiplication by $a$ is a one-to-one map on $\{0, 1, \dots, N-1\}$, and we leave the states $N \le y < 2^n$ unchanged. So $U_a$ is a permutation matrix, and building it is a problem in reversible logic.

We compute the permutation from the arithmetic, split it into disjoint cycles, and split each cycle into swaps of two states (transpositions). No step uses the order.

In [ ]:
def perm_of(m, N, n):
    """The permutation y -> (m*y) mod N on n qubits, with y >= N left fixed."""
    p = list(range(2 ** n))
    for y in range(N):
        p[y] = (m * y) % N
    return p


def cycles_of(p):
    """Disjoint non-trivial cycles of a permutation given as a list."""
    seen, out = set(), []
    for start in range(len(p)):
        if start in seen or p[start] == start:
            continue
        c, x = [], start
        while x not in seen:
            seen.add(x)
            c.append(x)
            x = p[x]
        out.append(c)
    return out


def transpositions_of(p):
    """
    Factor into transpositions, in the order a circuit must apply them.
    The cycle (c0 c1 ... ck) equals (c0 ck) o ... o (c0 c1) under right-to-left
    function composition, so the circuit applies (c0 c1) first.
    """
    out = []
    for c in cycles_of(p):
        for i in range(1, len(c)):
            out.append((c[0], c[i]))
    return out


p7 = perm_of(A, N, n)
print(f"y      -> {A}y mod {N}")
for y in range(N + 1):
    print(f"{y:>2}     -> {p7[y]:>2}")
print(f"\ncycles: {cycles_of(p7)}")
print(f"transpositions: {transpositions_of(p7)}")

The result is three cycles of length four and four fixed points. The fixed points are $0$, $5$, $10$ and the unused state $15$; $5$ is fixed because $7 \cdot 5 = 35 \equiv 5$, since $5$ shares a factor with $15$. None of this needed the order, although the cycle length turns out to equal it. Reading $r$ from a printed permutation only works because $N$ is small enough to print.

## Turning a transposition into gates

A swap of two basis states $\lvert x \rangle \leftrightarrow \lvert y \rangle$ becomes gates in three steps.

Let $d = x \oplus y$ and let $k$ be any bit position where $d$ is 1.

1. Apply a CNOT from qubit $k$ onto every other qubit where $d$ is 1. After this, $x$ and $y$ agree everywhere except bit $k$.
2. Apply a multi-controlled $X$ to qubit $k$, controlled on the other $n-1$ qubits matching the shared pattern. This flips bit $k$ for exactly those two states, which swaps them.
3. Undo the CNOTs.

The controlled version adds one control to the middle gate only. The CNOT layers undo each other, so when the control is off the whole block does nothing.

This costs one multi-controlled $X$ per transposition, and the number of transpositions grows with $2^n$. It is a lookup table and does not scale. Scalable constructions build modular multiplication from modular adders; Beauregard's uses $2n + 3$ qubits and $O(n^3)$ gates. We use the lookup table because at $n = 4$ it is easy to follow and to check exactly, and we compare its cost with the scalable ones later.

In [ ]:
def c_mult(m, N, n, label=None):
    """
    Controlled |y> -> |m*y mod N>.
    Qubit 0 is the control; qubits 1..n are the work register, qubit 1 the low bit.
    """
    qc = QuantumCircuit(1 + n, name=label or f"x{m} mod {N}")
    w = list(range(1, 1 + n))

    for x, y in transpositions_of(perm_of(m, N, n)):
        d = x ^ y
        k = (d & -d).bit_length() - 1          # lowest set bit of the difference
        fix = [j for j in range(n) if j != k and (d >> j) & 1]

        for j in fix:                           # collapse onto a single differing bit
            qc.cx(w[k], w[j])

        xp = x                                  # where x sits after that collapse
        for j in fix:
            xp ^= ((x >> k) & 1) << j

        ctrl_qubits = [j for j in range(n) if j != k]
        zeros = [w[j] for j in ctrl_qubits if not (xp >> j) & 1]

        for q in zeros:                         # controls fire on 0, so flip them
            qc.x(q)
        qc.mcx([0] + [w[j] for j in ctrl_qubits], w[k])
        for q in zeros:
            qc.x(q)

        for j in reversed(fix):                 # undo the collapse
            qc.cx(w[k], w[j])
    return qc


c_mult(A, N, n).draw('mpl', fold=110)

## Checking it against the arithmetic

This construction is easy to get slightly wrong in a way that still gives the right factors. Reversing the order of the transpositions builds multiplication by $a^{-1}$ instead of $a$. That has the same order, so the algorithm still returns $r = 4$ and still factors 15. The only way to catch it is to compare the circuit with the classical map, state by state.

In [ ]:
def check_multiplier(m):
    """Compare the controlled circuit's action against (m*y) mod N on every basis state."""
    op = Operator(c_mult(m, N, n)).data
    p = perm_of(m, N, n)
    for y in range(2 ** n):
        # qubit 0 is the control, so the amplitude index is control + 2*y
        if abs(op[1 + 2 * p[y], 1 + 2 * y] - 1) > 1e-9:      # control on: y -> m*y
            return False, y
        if abs(op[2 * y, 2 * y] - 1) > 1e-9:                  # control off: identity
            return False, y
    return True, None


for m in (2, 4, 7, 8, 11, 13):
    ok, bad = check_multiplier(m)
    gates = c_mult(m, N, n).count_ops()
    print(f"multiply by {m:>2}: matches arithmetic = {ok}"
          f"{'' if ok else f' (fails at y={bad})'}   "
          f"mcx = {gates.get('mcx', 0)}, cx = {gates.get('cx', 0)}, x = {gates.get('x', 0)}")

All six bases coprime to 15 build correctly. This check is what separates a correct implementation from a circuit that happens to give the expected answer.

## The multipliers, and a warning about 15

Phase estimation needs controlled $U_a^{2^j}$. Since $U_a^{2^j} = U_{a^{2^j} \bmod N}$, the multiplier can be computed classically by repeated squaring at no quantum cost. Every serious implementation does this, and it is a legitimate part of the algorithm.

For $N = 15$ it has an awkward consequence.

In [ ]:
t = 2 * n                              # counting register width, the textbook prescription
mults = [pow(A, 2 ** j, N) for j in range(t)]

print(f"counting register: t = {t} qubits (textbook prescription 2n for n = {n})")
print(f"multipliers a^(2^j) mod N: {mults}")
print(f"non-trivial (not equal to 1): {sum(m != 1 for m in mults)} of {t}")

Six of the eight controlled multiplications are multiplication by 1, which does nothing and costs nothing. This happens because 15 is small. The number of non-trivial multipliers is $\lceil \log_2 r \rceil$, and here $r = 4$. For a 2048-bit modulus with a period of similar size, all 4096 are non-trivial, and each is a full modular multiplication.

Every published factoring demonstration shares this feature. When a reported circuit is small, ask how much of that is the algorithm and how much is the particular number.

## Assembling order finding

This is phase estimation with the work register starting in $\lvert 1 \rangle$ and the controlled multipliers in place of a controlled unitary. The inverse QFT on the counting register turns the collected phases into a readable integer.

In [ ]:
def order_finding_circuit(t, a=A):
    """Phase estimation on U_a with a t-qubit counting register."""
    qc = QuantumCircuit(t + n, t)
    qc.h(range(t))                                  # uniform superposition of exponents
    qc.x(t)                                         # work register = |1>

    for j in range(t):
        m = pow(a, 2 ** j, N)
        if m == 1:                                  # identity, nothing to append
            continue
        qc.compose(c_mult(m, N, n), qubits=[j] + list(range(t, t + n)), inplace=True)

    qc.append(QFTGate(t).inverse(), range(t))       # read the phase
    qc.measure(range(t), range(t))
    return qc


honest = order_finding_circuit(t)
honest_t = transpile(honest, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

print(f"honest circuit: {honest.num_qubits} qubits")
print(f"  transpiled to {BASIS}: depth {honest_t.depth()}, "
      f"cx {honest_t.count_ops().get('cx', 0)}")

In [ ]:
SHOTS_SIM = 4096
counts_ideal = sim.run(transpile(honest, sim), shots=SHOTS_SIM).result().get_counts()

phases = {int(b, 2): c / SHOTS_SIM for b, c in counts_ideal.items()}

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.bar(list(phases), list(phases.values()), width=2.5, color='#a02580')
for s in range(r_true):
    ax.axvline(s * 2 ** t / r_true, color='k', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel(f'measured integer $k$ (counting register, $t$ = {t})')
ax.set_ylabel('probability')
ax.set_title('Ideal order-finding output: peaks at $k = s\\,2^t/r$')
ax.set_xlim(-4, 2 ** t)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("outcomes with probability above 1%:")
for k in sorted(phases):
    if phases[k] > 0.01:
        print(f"  k = {k:>3}   phase = {k / 2 ** t:.4f}   p = {phases[k]:.3f}")

There are four peaks, at $k/2^t = 0, 1/4, 1/2, 3/4$, each with probability about $1/4$. The dashed lines mark $s \cdot 2^t / r$ for the classically computed $r$, and the peaks sit on them exactly. They are exact because $r = 4$ divides $2^t$. For a period that does not divide the register size, the peaks spread out and continued fractions have more work to do.

## From a phase to a factor

Each outcome $k$ estimates $s/r$ for an unknown $s$. Continued fractions give the best fraction with denominator below $N$, and that denominator is the candidate period. Two of the four outcomes are useless: $k = 0$ carries no information, and $k = 2^{t-1}$ gives $s/r = 1/2$, whose denominator 2 fails because $7^2 = 4 \ne 1 \bmod 15$. So a single run succeeds about half the time, and the algorithm is run more than once.

In [ ]:
def factors_from_outcome(k, t, a=A):
    """Continued-fraction post-processing. Returns (r, factors, reason-if-failed)."""
    r = Fraction(k, 2 ** t).limit_denominator(N).denominator
    if r == 1:
        return r, None, 'denominator 1, no information'
    if pow(a, r, N) != 1:
        return r, None, f'a^{r} != 1 mod N, candidate period wrong'
    if r % 2:
        return r, None, 'odd period, reduction does not apply'
    x = pow(a, r // 2, N)
    if x == N - 1:
        return r, None, 'a^(r/2) = -1 mod N, reduction gives trivial factors'
    f = sorted({gcd(x - 1, N), gcd(x + 1, N)} - {1, N})
    return r, (f or None), None if f else 'gcd gave only trivial factors'


rows = []
for k in sorted(phases, key=lambda k: -phases[k])[:4]:
    r, f, why = factors_from_outcome(k, t)
    rows.append({'k': k, 'phase k/2^t': f'{k / 2 ** t:.3f}', 'p': f'{phases[k]:.3f}',
                 'candidate r': r, 'factors': ' x '.join(map(str, f)) if f else '-',
                 'outcome': 'success' if f else why})

pd.DataFrame(rows).set_index('k')

In [ ]:
success = sum(phases[k] for k in phases if factors_from_outcome(k, t)[1])
print(f"probability a single ideal run yields factors: {success:.3f}")
print(f"expected runs needed: {1 / success:.1f}")
print(f"probability of success within 5 runs: {1 - (1 - success) ** 5:.4f}")

## What a compiled demonstration does instead

Here is the shortcut. If you already know $r = 4$, the work register only needs to track position along the cycle $1 \to 7 \to 4 \to 13 \to 1$. That is four states, so two qubits. On that cycle, $U_a$ is just "add 1 modulo 4", which takes two gates.

The resulting circuit has the same output distribution as the honest one, so the same post-processing reports the factors 3 and 5. But it could not have been written without knowing the answer. We use a three-qubit counting register so that the ideal output has clear peaks, which helps when comparing with hardware.

In [ ]:
def compiled_circuit(t_c=3, r_known=None):
    """
    Phase estimation where U is an increment mod r on a compressed orbit register.
    Requires r in advance; that is the entire point.
    """
    r_known = r_known or r_true
    w = int(np.ceil(log2(r_known)))                    # 2 qubits for r = 4
    qc = QuantumCircuit(t_c + w, t_c)
    qc.h(range(t_c))

    for j in range(t_c):
        for _ in range(2 ** j % r_known):              # controlled increment mod 4
            qc.ccx(j, t_c, t_c + 1)                    # high bit ^= low bit
            qc.cx(j, t_c)                              # low bit ^= 1

    qc.append(QFTGate(t_c).inverse(), range(t_c))
    qc.measure(range(t_c), range(t_c))
    return qc


T_C = 3
compiled = compiled_circuit(T_C)
compiled_t = transpile(compiled, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

print(f"compiled circuit: {compiled.num_qubits} qubits, "
      f"depth {compiled_t.depth()}, cx {compiled_t.count_ops().get('cx', 0)}")
print(f"honest circuit:   {honest.num_qubits} qubits, "
      f"depth {honest_t.depth()}, cx {honest_t.count_ops().get('cx', 0)}")

counts_c = sim.run(transpile(compiled, sim), shots=SHOTS_SIM).result().get_counts()
print(f"\ncompiled ideal outcomes: "
      f"{sorted((int(b, 2), round(c / SHOTS_SIM, 3)) for b, c in counts_c.items())}")
print(f"post-processing on the compiled output: "
      f"{factors_from_outcome(2, T_C)[1]}")

The two circuits differ by two orders of magnitude in two-qubit gate count and report identical results. The honest circuit is the one that would still work if we did not know the answer, and it is the one current devices cannot run reliably.

## The same small circuit, applied to RSA-2048

If the compiled circuit only encodes the period, nothing ties it to $N = 15$. Any modulus with a base of order 4 gives the same circuit. So we build a 2048-bit RSA modulus, find a base of order 4 for it, and run the identical circuit.

Finding that base requires knowing the factors. That is the circularity, and the next cells carry it out.

In [ ]:
def is_prime(x, rounds=32, rng=random):
    """Miller-Rabin."""
    if x < 2:
        return False
    for sp in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if x % sp == 0:
            return x == sp
    d, s = x - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1
    for _ in range(rounds):
        y = pow(rng.randrange(2, x - 1), d, x)
        if y in (1, x - 1):
            continue
        for _ in range(s - 1):
            y = y * y % x
            if y == x - 1:
                break
        else:
            return False
    return True


def prime_1_mod_4(bits, rng):
    """A prime p = 1 mod 4, so that a square root of -1 exists modulo p."""
    while True:
        p = rng.getrandbits(bits) | (1 << (bits - 1)) | 1
        p += (1 - p % 4) % 4
        if p % 4 == 1 and is_prime(p, rng=rng):
            return p


rng = random.Random(2024)
p, q = prime_1_mod_4(1024, rng), prime_1_mod_4(1024, rng)
N_rsa = p * q

# a = (a square root of -1) mod p, and 1 mod q. Then ord(a) = 4 modulo N_rsa.
while True:
    z = pow(rng.randrange(2, p - 1), (p - 1) // 4, p)
    if z * z % p == p - 1:
        break
a_rsa = (z * q * pow(q, -1, p) + p * pow(p, -1, q)) % N_rsa

print(f"N is {N_rsa.bit_length()} bits, the size of an RSA-2048 modulus")
print(f"order of a modulo N is 4: {pow(a_rsa, 4, N_rsa) == 1 and pow(a_rsa, 2, N_rsa) != 1}")

In [ ]:
# The identical 4,000-shot run of the identical compiled circuit, reused verbatim.
r_rsa = None
for b in sorted(counts_c, key=lambda b: -counts_c[b]):
    cand = Fraction(int(b, 2), 2 ** T_C).limit_denominator(N_rsa).denominator
    if cand % 2 == 0 and pow(a_rsa, cand, N_rsa) == 1:
        r_rsa = cand
        break

x = pow(a_rsa, r_rsa // 2, N_rsa)
f1, f2 = gcd(x - 1, N_rsa), gcd(x + 1, N_rsa)

print(f"period recovered from the quantum output: r = {r_rsa}")
print(f"gcd(a^(r/2) - 1, N) is a {f1.bit_length()}-bit factor")
print(f"gcd(a^(r/2) + 1, N) is a {f2.bit_length()}-bit factor")
print(f"the two factors multiply back to N: {f1 * f2 == N_rsa}")
print(f"both are prime: {is_prime(f1, rng=rng) and is_prime(f2, rng=rng)}")

The measurement record of a five-qubit circuit has just split a 2048-bit number into its two 1024-bit prime factors. RSA is not broken. Choosing $a$ used $p$ and $q$ directly, and the period was short because we arranged it. This is Smolin, Smith and Vargo's argument in practice: the difficulty of a factoring demonstration depends on the length of the period found, not on the size of the number, and here $\log_2 r = 2$.

A useful way to read any factoring claim: what was the period, how many controlled multiplications were non-trivial, and was the base chosen before or after the factors were known?

## On real devices

Both circuits now run on real devices. The honest circuit with $t = 8$ needs 12 qubits, so for hardware we use $t = 4$. That works for this instance only because $r = 4$ divides 16 exactly, which is itself a fact about the answer. It saves four qubits and very little depth, because the cost is in the two controlled multiplications.

| Device | Vendor | Type | Relevant here |
|---|---|---|---|
| `rigetti:rigetti:qpu:cepheus-1-108q` | Rigetti | Superconducting, square lattice | Higher two-qubit error than Garnet in recent calibrations; circuit depth is the limit |
| `aws:iqm:qpu:garnet` | IQM | Superconducting, square lattice | Multi-controlled gates need routing |
| `aws:aqt:qpu:ibex-q1` (optional) | AQT | Trapped ion, all qubits connected | No routing needed; runs in scheduled windows and costs more per shot |

AQT is commented out in the next cell; uncomment it to include it. The default run is 2 jobs per device and may take minutes to hours depending on queues.

In [ ]:
honest_hw = order_finding_circuit(4)                      # 8 qubits, fits every device
honest_hw_t = transpile(honest_hw, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)

CIRCUITS = {
    'Honest (t=4)':  honest_hw,
    'Compiled (t=3)': compiled,
}

# Ideal support: the outcomes the circuit should produce, for scoring the hardware runs.
IDEAL_SUPPORT = {
    'Honest (t=4)':   {s * 2 ** 4 // r_true for s in range(r_true)},
    'Compiled (t=3)': {s * 2 ** T_C // r_true for s in range(r_true)},
}

for label, qc in CIRCUITS.items():
    tq = transpile(qc, basis_gates=BASIS, optimization_level=2, seed_transpiler=7)
    print(f"{label:16s} {qc.num_qubits} qubits, depth {tq.depth():>5}, "
          f"cx {tq.count_ops().get('cx', 0):>4}, ideal support {sorted(IDEAL_SUPPORT[label])}")
print(f"{'Honest (t=8)':16s} {honest.num_qubits} qubits, depth {honest_t.depth():>5}, "
      f"cx {honest_t.count_ops().get('cx', 0):>4}   (too wide for some devices)")

In [ ]:
provider = QbraidProvider()

# Devices are named by qBraid QRN. The README lists devices, prices and availability.
BACKENDS = {
    'Rigetti Cepheus': 'rigetti:rigetti:qpu:cepheus-1-108q',
    'IQM Garnet':      'aws:iqm:qpu:garnet',
    # 'AQT IBEX Q1':   'aws:aqt:qpu:ibex-q1',   # trapped ion; runs in scheduled windows; 2.35 credits per shot
}

COLORS = {
    'Rigetti Cepheus': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT IBEX Q1':     '#2d7a4f',
}


devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}
print("Configured backends:", list(devices))

In [ ]:
hw_counts = {name: {} for name in BACKENDS}

for name, device in devices.items():
    for label, qc in CIRCUITS.items():
        job = device.run(qc, shots=SHOTS, tags=QUEST_JOB_TAGS)
        hw_counts[name][label] = job.result().data.get_counts()
        on_support = sum(c for b, c in hw_counts[name][label].items()
                         if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS
        print(f"{name:18s} {label:16s} shots on ideal support: {on_support:.3f}")

## Hardware results

Both panels use the same devices and shot count. The dashed line shows what a device producing only noise would give: a uniform distribution over the counting register, which puts $r/2^t$ of its weight on the correct outcomes by chance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)

for ax, (label, qc) in zip(axes, CIRCUITS.items()):
    t_lab = qc.num_clbits
    chance = len(IDEAL_SUPPORT[label]) / 2 ** t_lab

    names = list(BACKENDS)
    vals = [sum(c for b, c in hw_counts[nm][label].items()
                if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS for nm in names]

    ax.bar(range(len(names)), vals, color=[COLORS[nm] for nm in names], width=0.6)
    ax.axhline(1.0, color='k', linestyle='--', linewidth=2, alpha=0.6, label='Ideal')
    ax.axhline(chance, color='gray', linestyle=':', linewidth=2,
               label=f'Depolarised ({chance:.2f})')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([nm.replace(' ', '\n') for nm in names], fontsize=9)
    ax.set_title(f'{label}\n{qc.num_qubits} qubits, '
                 f'{transpile(qc, basis_gates=BASIS, optimization_level=2, seed_transpiler=7).count_ops().get("cx", 0)} CX')
    ax.set_ylim(0, 1.12)
    ax.grid(alpha=0.3, axis='y')
    ax.legend(fontsize=9, loc='upper right')

axes[0].set_ylabel('fraction of shots on the ideal support')
fig.suptitle('Order finding for $N = 15$: honest circuit against compiled circuit', y=1.0)
plt.tight_layout()
plt.show()

We expect the compiled circuit to land near the ideal line and the honest circuit near the noise line. In our tests on Rigetti Cepheus, the compiled circuit put 0.48 of its shots on correct outcomes, against 0.25 for pure noise, while the honest circuit gave 0.22, slightly below noise.

If the honest circuit lands at the noise level, the device told us nothing about the period. Any factors recovered from that run came from the post-processing, not from the quantum computation. Post-processing that tries several outcomes will eventually hit a usable one even on pure noise, because $N = 15$ has only four candidate periods. A report that "3 and 5 were recovered", without the output distribution, does not show that the circuit worked.

## Summary

In [ ]:
rows = []
for name in BACKENDS:
    for label, qc in CIRCUITS.items():
        t_lab = qc.num_clbits
        counts_hw = hw_counts[name][label]
        chance = len(IDEAL_SUPPORT[label]) / 2 ** t_lab
        on_support = sum(c for b, c in counts_hw.items()
                         if int(b, 2) in IDEAL_SUPPORT[label]) / SHOTS
        best = max(counts_hw, key=counts_hw.get)
        r_hw, f_hw, _ = factors_from_outcome(int(best, 2), t_lab)
        rows.append({
            'Backend': name,
            'Circuit': label,
            'CX': transpile(qc, basis_gates=BASIS, optimization_level=2,
                            seed_transpiler=7).count_ops().get('cx', 0),
            'On ideal support': f'{on_support:.3f}',
            'Chance level': f'{chance:.3f}',
            'Signal above chance': f'{(on_support - chance) / (1 - chance):+.3f}',
            'Most likely k': int(best, 2),
            'Factors from it': ' x '.join(map(str, f_hw)) if f_hw else '-',
        })

pd.DataFrame(rows).set_index(['Backend', 'Circuit'])

The "signal above chance" column rescales each result so that 1.0 is an ideal device and 0.0 is pure noise. The "factors from it" column is the one a press release would quote. Comparing the two columns shows why that is misleading.

## What factoring RSA-2048 would cost

Extrapolating from 15 tells us little, so we use published estimates. Gidney and Ekera (2019) give resource counts for factoring an $n$-bit RSA modulus on a surface-code machine:

$$\text{logical qubits} = 3n + 0.002\, n \log_2 n, \qquad \text{Toffoli gates} = 0.3\, n^3 + 0.0005\, n^3 \log_2 n.$$

For $n = 2048$ their physical estimate is **about 20 million noisy qubits running for 8 hours**. This assumes a physical gate error rate of $10^{-3}$, a surface-code cycle time of 1 microsecond, a reaction time of 10 microseconds, and a planar grid with nearest-neighbour connections.

Gidney (2025) revisits the problem under the same hardware assumptions and arrives at **fewer than 1 million noisy qubits running for less than a week**. The reduction in qubits comes from better algorithms, paid for partly with a longer runtime. Nothing about the assumed hardware changed.

In [ ]:
def logical_qubits(nbits):
    return 3 * nbits + 0.002 * nbits * log2(nbits)


def toffoli_count(nbits):
    return 0.3 * nbits ** 3 + 0.0005 * nbits ** 3 * log2(nbits)


sizes = [4, 15, 128, 512, 1024, 2048, 4096]
est = pd.DataFrame({
    'n (bits)': sizes,
    'logical qubits': [f'{logical_qubits(s):,.0f}' for s in sizes],
    'Toffoli gates': [f'{toffoli_count(s):.2e}' for s in sizes],
})
print(est.to_string(index=False))

print(f"\nFor N = 15 (n = {n}): {logical_qubits(n):.0f} logical qubits, "
      f"{toffoli_count(n):.0f} Toffolis.")
print(f"Our lookup-table circuit used {honest_t.count_ops().get('cx', 0)} CX gates, "
      f"about {honest_t.count_ops().get('cx', 0) / (6 * toffoli_count(n)):.1f}x the "
      f"CX cost of {toffoli_count(n):.0f} Toffolis at 6 CX each.")

These formulas are fits for large $n$ and were never meant for $n = 4$, so treat the comparison at $n = 4$ as a rough order of magnitude. It still shows the main point: the lookup-table construction is already more expensive at the smallest size, and its cost doubles with every added bit, while the fitted cost grows as $n^3$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

# left: how the two costs scale
ns = np.array([2 ** k for k in range(2, 13)])
axes[0].loglog(ns, [logical_qubits(x) for x in ns], 'o-', color='#1a5285',
               linewidth=2.5, markersize=7, label='Logical qubits')
axes[0].loglog(ns, [toffoli_count(x) for x in ns], 's-', color='#a02580',
               linewidth=2.5, markersize=7, label='Toffoli gates')
axes[0].axvline(2048, color='k', linestyle='--', linewidth=2, alpha=0.6)
axes[0].text(2048, 3e2, ' RSA-2048', rotation=90, va='bottom', fontsize=10)
axes[0].axvline(n, color='#2d7a4f', linestyle=':', linewidth=2)
axes[0].text(n, 3e2, ' N = 15', rotation=90, va='bottom', fontsize=10, color='#2d7a4f')
axes[0].set_xlabel('modulus size $n$ (bits)')
axes[0].set_ylabel('count')
axes[0].set_title('Gidney-Ekera scaling')
axes[0].grid(alpha=0.3, which='both')
axes[0].legend()

# right: the physical estimate, and what changed between 2019 and 2025
labels = ['Gidney-Ekera\n2019', 'Gidney\n2025']
qubits = [20e6, 1e6]
bars = axes[1].bar(labels, qubits, color=['#1a5285', '#a02580'], width=0.55)
axes[1].set_yscale('log')
axes[1].set_ylabel('physical qubits for RSA-2048')
axes[1].set_title('Same hardware assumptions, six years apart')
axes[1].grid(alpha=0.3, axis='y', which='both')
for bar, val, rt in zip(bars, qubits, ['8 hours', '< 1 week']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val * 1.15,
                 f'{val:,.0f}\n{rt}', ha='center', fontsize=10)
axes[1].set_ylim(1e5, 1e8)

plt.tight_layout()
plt.show()

print(f"largest device on qBraid today: order 10^2 physical qubits")
print(f"shortfall against the 2025 estimate: {1e6 / 100:,.0f}x")

## Where this leaves the field

Coverage of quantum factoring often mixes up three separate questions.

**What has been factored without shortcuts.** The largest number factored without prior knowledge of the answer is 21, and that implementation also took simplifications. Larger results in the literature are either compiled, or use a different algorithm. The 56153 result often cited alongside them used an adiabatic minimisation, not Shor's algorithm, so Shor's scaling does not apply to it. This record has changed little in over a decade, and the hardware section above shows why.

**What the hardware would need.** Under a million physical qubits at $10^{-3}$ gate error, running for days behind a surface code. Current devices have one thousand to ten thousand times fewer qubits and are not error-corrected at the required level.

**Why the deadline comes before the machine.** Encrypted traffic recorded today can be stored and decrypted once such a machine exists. Mosca's inequality states the problem: if data must stay secret for $x$ years, migration takes $y$ years, and a capable machine is $z$ years away, you are already late when $x + y > z$. For data that must stay secret for twenty years, with a ten-year migration, that condition holds for most published estimates of $z$.

The response is already standardised. NIST published FIPS 203 (ML-KEM), FIPS 204 (ML-DSA) and FIPS 205 (SLH-DSA) on 13 August 2024, selected HQC as a backup key-encapsulation mechanism in March 2025, and has FN-DSA in draft as FIPS 206. NIST IR 8547 sets the transition schedule: RSA-2048 and ECC P-256 deprecated by 2030 and disallowed after 2035.

To summarise: the algorithm works, the machine does not yet exist, the resource estimates for it have fallen by a factor of about twenty in six years without any change in assumed hardware, and migration is under way on a schedule set by that uncertainty. Factoring 15 on a small device does not change any of this.

## Going further

- **Use a period that does not divide the register.** Factor $N = 21$ with $a = 2$, where $r = 6$ does not divide $2^t$. The peaks spread out, continued fractions do real work, and each run succeeds less often. Every real instance is like this.
- **Replace the lookup table with an adder.** Implement Beauregard's modular multiplier on $2n + 3$ qubits and compare its gate count with `c_mult` at $n = 4, 5, 6$. Find where the adder becomes cheaper, and compare with the $n^3$ fit above.
- **Try every base.** Run the honest circuit for all six $a$ coprime to 15 and record the period, whether the reduction succeeds, and the circuit cost. Two of the six give $r = 2$ and a much cheaper circuit. Would reporting only those be a fair summary?
- **Assess a published claim.** Take a published quantum factoring result and find three numbers: the period found, the number of non-trivial controlled multiplications, and whether the base was chosen before the factors were known. Then say what the result demonstrates.
- **Estimate a threat timeline.** Put an organisation's numbers into Mosca's inequality, using the 2025 resource estimate and a hardware roadmap you find credible, and work out when migration needs to start.